# Drone Tracker v5 yolov8m + imgsz=1280 (small-drone fix)

**Empiryczna motywacja** (z lokalnych pomiarow):
- v4 dataset: 64.5% bboxow <30 px (median 12 px native)
- runtime imgsz=640 + 1920x1080 video skaluje drona 12 px do 4 px
  (ponizej yolov8 stride 8) -> YOLO blindness na male drony
- v4@960 testowane: recall 18.8% -> 19.6%, mean conf +87% — niewystarczajace
- v5 cel: yolov8m (28M params) + imgsz=1280 (drone 12 px -> 8 px po scale)

**Wymagania**:
- Colab Pro (T4 16 GB minimum, lepiej V100/A100)
- v4_dataset.zip na Drive: MyDrive/drone_tracker/v4_dataset.zip
- ETA: T4 ~10-14h, V100 ~6-8h

**Output (Drive)**: MyDrive/drone_tracker/v5/v5_drone_m_imgsz1280/
  - best.pt
  - v5_best_imgsz1280.onnx (FP32)
  - v5_best_fp16_imgsz1280.onnx (FP16, default w pipeline)


## Krok 1 — Weryfikacja GPU

In [ ]:
!nvidia-smi

Powinno pokazac T4 (16 GB) lub V100/A100 (Pro). yolov8m @ imgsz=1280 batch=16:
- T4 (16 GB) — OK z batch=16
- V100 / A100 (40 GB) — batch=24-32 dla 30% szybszego trainingu

Jesli OOM, zmniejsz batch w komorce trening do 8.

## Krok 2 — Instalacja

In [ ]:
!pip install -q ultralytics==8.4.30 onnx onnxslim

In [ ]:
import torch
from ultralytics import YOLO
print(f'torch={torch.__version__}, cuda={torch.cuda.is_available()}, '
      f'device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}')


## Krok 3 — Mount Drive + upload v5 dataset

Dataset v5 = kopia v4 (4032 obrazki, ten sam content, inny layout: training/v5/).

**Lokalnie przed Colab**:
```bash
python tools/_build_v5_dataset.py        # tworzy training/v5/ + training/v5_dataset.zip (~1 GB)
```

Potem drag&drop `training/v5_dataset.zip` do `MyDrive/drone_tracker/`.

Schemat: jeden plik `vN_dataset.zip` per wersja, jedna sciezka, brak konfuzji.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/drone_tracker/v5'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive output: {DRIVE_BASE}')


In [ ]:
import shutil, os
from pathlib import Path

ZIP_PATH = '/content/drive/MyDrive/drone_tracker/v5_dataset.zip'
WORK_DIR = '/content/v5_work'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f'Brak {ZIP_PATH}. Wgraj v5_dataset.zip na Drive.\n'
        f'Lokalnie odpal: python tools/_build_v5_dataset.py'
    )

os.makedirs(WORK_DIR, exist_ok=True)
shutil.unpack_archive(ZIP_PATH, WORK_DIR)
print(f'Rozpakowano do {WORK_DIR}')
!ls {WORK_DIR}/training/v5/


## Krok 4 — Verify dataset

In [ ]:
import os
DATA_YAML = f'{WORK_DIR}/training/v5/data.yaml'
TRAIN_IMGS = f'{WORK_DIR}/training/v5/images/train'
VAL_IMGS = f'{WORK_DIR}/training/v5/images/val'

print('data.yaml:')
print(open(DATA_YAML).read())

n_train = sum(1 for f in os.listdir(TRAIN_IMGS) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
n_val = sum(1 for f in os.listdir(VAL_IMGS) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
print(f'\nimages: train={n_train}  val={n_val}  total={n_train + n_val}')
print(f'oczekiwane: train=3225  val=807  total=4032 (z v4 source)')


## Krok 5 — Trening v5 yolov8m @ imgsz=1280

**Konfiguracja vs v4** (zmiany dla small-drone fix):
- base: yolov8s.pt -> **yolov8m.pt** (28M params)
- imgsz: 640 -> **1280** (drone 12 px native -> 8 px po scale)
- batch: 32 -> **16** (yolov8m@1280 VRAM-ciezki)
- epochs: 50 -> **60**
- patience: 15 -> **20**
- copy_paste: 0.0 -> **0.3** (KLUCZOWE dla small obj recall)
- mixup: 0.15 -> 0.10

**ETA T4**: ~10-14h. **V100/A100**: ~6-8h.

In [ ]:
from ultralytics import YOLO
import os

RUN_NAME = 'v5_drone_m_imgsz1280'
RUN_PROJECT = '/content/runs'

model = YOLO('yolov8m.pt')
model.train(
    data=DATA_YAML,
    imgsz=1280,
    epochs=60,
    batch=16,
    device=0,
    patience=20,
    name=RUN_NAME,
    project=RUN_PROJECT,

    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    cos_lr=True,

    mosaic=1.0,
    mixup=0.10,
    copy_paste=0.3,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    translate=0.1,
    perspective=0.0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.3,
    close_mosaic=10,

    plots=True, save=True, verbose=True,
)

best = f'{RUN_PROJECT}/{RUN_NAME}/weights/best.pt'
print(f'\n[v5] best weights: {best}')


## Krok 6 — Eksport ONNX FP32 + FP16

In [ ]:
m = YOLO(best)
fp32_path = m.export(format='onnx', imgsz=1280, opset=12, simplify=False, dynamic=False)
fp32_target = f'{os.path.dirname(best)}/v5_best_imgsz1280.onnx'
if os.path.exists(fp32_path) and fp32_path != fp32_target:
    os.rename(fp32_path, fp32_target)
print(f'FP32 ONNX: {fp32_target} ({os.path.getsize(fp32_target)/1e6:.1f} MB)')

m = YOLO(best)
fp16_path = m.export(format='onnx', imgsz=1280, opset=12, simplify=False, dynamic=False, half=True)
fp16_target = f'{os.path.dirname(best)}/v5_best_fp16_imgsz1280.onnx'
if os.path.exists(fp16_path) and fp16_path != fp16_target:
    os.rename(fp16_path, fp16_target)
print(f'FP16 ONNX: {fp16_target} ({os.path.getsize(fp16_target)/1e6:.1f} MB)')


## Krok 7 — Eval na val set

In [ ]:
m = YOLO(best)
print(f'\n=== Eval {RUN_NAME} @ imgsz=1280 ===')
metrics = m.val(data=DATA_YAML, imgsz=1280, batch=8, plots=False, verbose=False)
print(f'  mAP@0.5      = {metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95 = {metrics.box.map:.4f}')
print(f'  precision    = {metrics.box.mp:.4f}')
print(f'  recall       = {metrics.box.mr:.4f}')
print(f'\nv4 mAP@0.5 baseline = 0.956 (z commit 8cbe953)')
print(f'v5 lepiej? {metrics.box.map50 > 0.956}')


## Krok 8 — Skopiuj caly run folder na Drive

Parity z v4 notebook: kopiujemy CALY folder runu (best.pt, last.pt, args.yaml, results.csv,
plots, train_batch*.jpg, val_batch*.jpg, weights/) — nie tylko ONNX-y.

Dlaczego: best.pt potrzebny do re-export INT8 (D5 NPU XDNA2) bez powtarzania
14h treningu; results.csv + plots do post-mortem analizy konwergencji.

In [ ]:
import shutil

DST = f'{DRIVE_BASE}/{RUN_NAME}'
SRC = f'{RUN_PROJECT}/{RUN_NAME}'

print(f'Kopiuje CALY folder runu: {SRC} -> {DST}')
print(f'(ETA 2-5 min na Drive, ~150-300 MB total)')
shutil.copytree(SRC, DST, dirs_exist_ok=True)

# Sanity check — czy kluczowe pliki dotarly
expected = [
    'weights/best.pt',
    'weights/v5_best_imgsz1280.onnx',
    'weights/v5_best_fp16_imgsz1280.onnx',
    'results.csv',
    'args.yaml',
]
print('\nSanity check:')
for fname in expected:
    path = f'{DST}/{fname}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'  OK   {fname}  ({size/1e6:.1f} MB)')
    else:
        print(f'  MISS {fname}')

print(f'\nDone. Pobierz CALY folder z Drive: {DST}')
print(f'  -> lokalnie: data/weights/{RUN_NAME}/')


## Krok 9 — Pobierz weights lokalnie

Na lokalnym kompie pobierz **caly folder** z Drive:
`MyDrive/drone_tracker/v5/v5_drone_m_imgsz1280/`

Zapisz do:
```
C:\dev\drone-tracker-system\data\weights\v5_drone_m_imgsz1280\
  ├─ weights/
  │   ├─ best.pt
  │   ├─ v5_best_imgsz1280.onnx           (FP32, ~210 MB)
  │   └─ v5_best_fp16_imgsz1280.onnx      (FP16, ~110 MB) <- runtime model
  ├─ results.csv
  ├─ args.yaml
  └─ *.png (training curves)
```

**Default model w main.cpp** (do zmiany po pobraniu):
```cpp
// cpp/app/main.cpp linia ~51:
std::string model = "../../../data/weights/v5_drone_m_imgsz1280/weights/v5_best_fp16_imgsz1280.onnx";
int imgsz = 1280;
```

**Test w pipeline**:
```bash
./cpp/build/Release/dtracker_main.exe \
  --video artifacts/test_videos/video_test_wide.mp4 \
  --model data/weights/v5_drone_m_imgsz1280/weights/v5_best_fp16_imgsz1280.onnx \
  --imgsz 1280 \
  --max-frames 25000 --no-gui --no-record
```

**Walidacja vs v4**:
- `python tools/eval_imgsz_comparison.py` (update MODELS dla v5)
- `python tools/analyze_ghost_tracks.py artifacts/runs/<v5_run>/telemetry.jsonl`

**Cele**:
- LOCKED na 25000 klatek: 24.8% (v4@640) -> 60%+ (v5@1280)
- id=42 detection rate: 19% -> 50%+
- B_KALMAN ghosts: 56.9% -> <30%
